# Eastern Cape Historical Election Pipeline

## Phase 1 — Step 3: Standardise Historical IEC Election Data (2000–2021)

**Objective:** standardise Eastern Cape LGE data for 2000, 2006, 2011, 2016 and 2021 into one consistent row-level schema before municipality-boundary harmonisation.

This notebook deliberately **does not harmonise historical municipality boundaries yet** and does not calculate final municipality-level ENP/margins. Those belong to Step 4 after the geographic crosswalk is fixed.

### Target schema
`Year, Province, MuniCode, Municipality, Ward, VotingDistrict, BallotType, RegisteredVoters, ValidVotes, SpoiltVotes, VotesCast, PartyName, PartyVotes, Turnout`


In [ ]:
from pathlib import Path
import re, warnings
import numpy as np
import pandas as pd
from IPython.display import display
pd.set_option("display.max_columns",100)
print("pandas", pd.__version__)


## 1. Project paths
The notebook searches upward for the repository folder containing `data/`. If needed, use the commented fallback path.


In [ ]:
def find_project_root(start=None):
    start=Path(start or Path.cwd()).resolve()
    for p in [start,*start.parents]:
        if (p/'data').exists(): return p
    return start
PROJECT_ROOT=find_project_root()
RAW_DIR=PROJECT_ROOT/'data'/'raw'
STANDARD_DIR=PROJECT_ROOT/'data'/'processed'/'standardized_elections'
STANDARD_DIR.mkdir(parents=True,exist_ok=True)
print('PROJECT_ROOT:',PROJECT_ROOT)
print('RAW_DIR:',RAW_DIR)
print('STANDARD_DIR:',STANDARD_DIR)


In [ ]:
# Fallback only if auto-detection is wrong:
# PROJECT_ROOT = Path(r"C:\Users\Monwabisi05\IdeaProjects\Dirisa_TeamQualification")
# RAW_DIR = PROJECT_ROOT/'data'/'raw'
# STANDARD_DIR = PROJECT_ROOT/'data'/'processed'/'standardized_elections'
# STANDARD_DIR.mkdir(parents=True,exist_ok=True)


## 2. Discover candidate EC election files


In [ ]:
csv_files=sorted(RAW_DIR.rglob('*.csv')) if RAW_DIR.exists() else []
print(f'CSV files found: {len(csv_files)}')
for p in csv_files: print(p.relative_to(PROJECT_ROOT))


In [ ]:
YEARS=[2000,2006,2011,2016,2021]
def score_candidate(path,year):
    s=str(path).lower().replace('\\','/'); name=path.name.lower(); score=0
    if str(year) in s: score+=10
    if name in {'ec.csv',f'ec_{year}.csv',f'ec ({year}).csv'}: score+=10
    if 'ec' in name: score+=4
    if 'detailed' in s: score+=4
    if 'result' in s: score+=2
    if 'turnout' in s: score-=5
    if 'summary' in s: score-=5
    if 'processed' in s or 'clean' in s: score-=4
    return score
candidate_files={}
for year in YEARS:
    ranked=sorted([(score_candidate(p,year),p) for p in csv_files],key=lambda x:x[0],reverse=True)
    ranked=[x for x in ranked if x[0]>0]
    candidate_files[year]=ranked[:10]
    print(f'\n--- {year} candidates ---')
    for score,p in ranked[:10]: print(score,p.relative_to(PROJECT_ROOT))


### Select source files
The highest-scoring candidate is selected automatically. **Check the printed paths.** If a year is wrong, set only that entry in `FILE_OVERRIDES`.


In [ ]:
FILE_OVERRIDES={
# 2000: RAW_DIR/'detailed'/'2000 LGE.csv',
# 2006: RAW_DIR/'detailed'/'2006 LGE.csv',
# 2011: RAW_DIR/'detailed'/'2011_detailed'/'EC.csv',
# 2016: RAW_DIR/'detailed'/'2016_detailed'/'EC.csv',
# 2021: RAW_DIR/'detailed'/'2021_detailed'/'EC.csv',
}
selected_files={}
for year in YEARS:
    selected_files[year]=Path(FILE_OVERRIDES[year]) if year in FILE_OVERRIDES else (candidate_files[year][0][1] if candidate_files[year] else None)
for year,p in selected_files.items(): print(year,'->',p.relative_to(PROJECT_ROOT) if p else 'NOT FOUND')


## 3. Load and inspect raw schemas


In [ ]:
def read_csv_robust(path):
    errs=[]
    for enc in ('utf-8-sig','utf-8','cp1252','latin1'):
        try: return pd.read_csv(path,encoding=enc,low_memory=False)
        except Exception as e: errs.append((enc,repr(e)))
    raise RuntimeError(errs)
def inspect_df(df,year,n=3):
    print('='*80,f'\n{year} RAW DATA\n','='*80)
    print('Shape:',df.shape); print('Columns:',list(df.columns)); display(df.head(n))
raw={}
for year,p in selected_files.items():
    if p is None or not p.exists(): print(year,'unavailable'); continue
    try:
        raw[year]=read_csv_robust(p); inspect_df(raw[year],year)
    except Exception as e: print(year,'LOAD FAILED:',e)


## 4. Resolve different IEC column names


In [ ]:
def key(text): return re.sub(r'[^a-z0-9]+','',str(text).strip().lower())
COLUMN_ALIASES={
'Province':['Province','ProvinceName'],
'Municipality':['Municipality','MunicipalityName','MunicipalArea','MunicipalAreaName','LocalMunicipality','MunicipalName'],
'Ward':['Ward','WardID','WardNumber','WardNo'],
'VotingDistrict':['VotingDistrict','Voting District','VotingDistrictNumber','VD','VDNumber'],
'RegisteredVoters':['RegisteredVoters','Registered Voters','Registered','RegVoters'],
'BallotType':['BallotType','Ballot Type','Ballot'],
'SpoiltVotes':['SpoiltVotes','Spoilt Votes','Spoilt','SpoiledVotes','Spoiled Votes'],
'PartyName':['PartyName','Party Name','Party','PoliticalParty'],
'PartyVotes':['TotalValidVotes','Total Valid Votes','PartyVotes','Party Votes','Votes','ValidVotesForParty'],
'ValidVotes':['ValidVotesCast','Valid Votes Cast','TotalValidVotesCast','Total Valid Votes Cast'],
'VotesCast':['TotalVotesCast','Total Votes Cast','VotesCast','Votes Cast','BallotsCast','Ballots Cast'],
'Turnout':['%VoterTurnout','% Voter Turnout','VoterTurnout','Voter Turnout','Turnout','Turnout%']}
def resolve_column(df,aliases):
    lookup={key(c):c for c in df.columns}
    for a in aliases:
        if key(a) in lookup: return lookup[key(a)]
    return None
def resolved_schema(df): return {t:resolve_column(df,a) for t,a in COLUMN_ALIASES.items()}
for year,df in raw.items():
    print('\n',year); display(pd.DataFrame(resolved_schema(df).items(),columns=['Standard field','Raw column']))


## 5. Municipality code extraction


In [ ]:
MUNI_CODE_PATTERN=re.compile(r'^\s*([A-Za-z]{2,3}\d{0,3})\s*[-–—]\s*(.+?)\s*$')
def split_municipality(value):
    if pd.isna(value): return pd.Series([pd.NA,pd.NA],index=['MuniCode','Municipality'])
    text=str(value).strip(); m=MUNI_CODE_PATTERN.match(text)
    if m: return pd.Series([m.group(1).upper(),m.group(2).strip()],index=['MuniCode','Municipality'])
    if re.fullmatch(r'(?:EC\d{3}|BUF|NMA)',text.upper()): return pd.Series([text.upper(),pd.NA],index=['MuniCode','Municipality'])
    return pd.Series([pd.NA,text],index=['MuniCode','Municipality'])


## 6. Standardise each election


In [ ]:
STANDARD_COLUMNS=['Year','Province','MuniCode','Municipality','Ward','VotingDistrict','BallotType','RegisteredVoters','ValidVotes','SpoiltVotes','VotesCast','PartyName','PartyVotes','Turnout']
def numeric(s):
    if s is None: return None
    return pd.to_numeric(s.astype('string').str.replace(',','',regex=False).str.replace('%','',regex=False).str.strip(),errors='coerce')
def standardise_election(df,year):
    schema=resolved_schema(df); out=pd.DataFrame(index=df.index); out['Year']=int(year)
    for target in ['Province','Ward','VotingDistrict','BallotType','PartyName']:
        src=schema.get(target); out[target]=df[src] if src else pd.NA
    src=schema.get('Municipality')
    if src:
        parts=df[src].apply(split_municipality); out['MuniCode']=parts['MuniCode']; out['Municipality']=parts['Municipality']
    else: out['MuniCode']=pd.NA; out['Municipality']=pd.NA
    for target in ['RegisteredVoters','ValidVotes','SpoiltVotes','VotesCast','PartyVotes','Turnout']:
        src=schema.get(target); out[target]=numeric(df[src]) if src else np.nan
    mask=out['VotesCast'].isna() & out['ValidVotes'].notna() & out['SpoiltVotes'].notna()
    out.loc[mask,'VotesCast']=out.loc[mask,'ValidVotes']+out.loc[mask,'SpoiltVotes']
    mask=out['Turnout'].isna() & out['RegisteredVoters'].gt(0) & out['VotesCast'].notna()
    out.loc[mask,'Turnout']=out.loc[mask,'VotesCast']/out.loc[mask,'RegisteredVoters']*100
    for c in ['Province','MuniCode','Municipality','Ward','VotingDistrict','BallotType','PartyName']:
        out[c]=out[c].astype('string').str.strip().replace({'':pd.NA,'nan':pd.NA,'None':pd.NA})
    out['Province']=out['Province'].replace({'EC':'Eastern Cape','EASTERN CAPE':'Eastern Cape','Eastern cape':'Eastern Cape'})
    return out[STANDARD_COLUMNS]
standardised={}
for year,df in raw.items():
    try:
        standardised[year]=standardise_election(df,year)
        print(year,df.shape,'->',standardised[year].shape); display(standardised[year].head(3))
    except Exception as e: print(year,'STANDARDISATION FAILED:',e)


## 7. Quality checks


In [ ]:
def quality_report(df,year):
    return {'Year':year,'Rows':len(df),'Muni non-null %':round(df.Municipality.notna().mean()*100,2),'MuniCode non-null %':round(df.MuniCode.notna().mean()*100,2),'Registered non-null %':round(df.RegisteredVoters.notna().mean()*100,2),'Party non-null %':round(df.PartyName.notna().mean()*100,2),'PartyVotes non-null %':round(df.PartyVotes.notna().mean()*100,2),'Turnout<0':int((df.Turnout<0).sum()),'Turnout>100':int((df.Turnout>100).sum()),'NegativePartyVotes':int((df.PartyVotes<0).sum())}
quality=pd.DataFrame([quality_report(df,y) for y,df in standardised.items()]).sort_values('Year')
display(quality)
for y,df in standardised.items(): assert list(df.columns)==STANDARD_COLUMNS,f'{y}: schema mismatch'
print('Common schema check passed.')


## 8. Inspect ballot types and municipality identifiers


In [ ]:
for year,df in standardised.items():
    print('\n'+'='*70); print(year); print('='*70)
    print('Ballot types:'); print(df.BallotType.value_counts(dropna=False).head(20))
    print('\nMunicipality count:',df[['MuniCode','Municipality']].drop_duplicates().shape[0])
    display(df[['MuniCode','Municipality']].drop_duplicates().sort_values(['MuniCode','Municipality'],na_position='last').head(50))


## 9. Save intermediate standardised files


In [ ]:
saved=[]
for year,df in standardised.items():
    p=STANDARD_DIR/f'ec_lge_{year}_standardized.csv'; df.to_csv(p,index=False); saved.append(p); print('Saved:',p.relative_to(PROJECT_ROOT))
for p in saved:
    check=pd.read_csv(p,low_memory=False); missing=[c for c in STANDARD_COLUMNS if c not in check.columns]; assert not missing,(p,missing)
print('Reload/schema test passed.')


# Step 3 completion checklist
- Verify the selected raw file for each year.
- Investigate any impossible turnout/vote values.
- Confirm ballot types.
- Inspect municipality-code coverage.
- Confirm all five outputs save successfully.

## Next: Phase 1 — Step 4, Municipality Harmonisation
Build an explicit historical municipality crosswalk; map/merge historical units into the chosen comparable Eastern Cape geography; document dropped units and boundary changes; validate coverage by year; and only then aggregate final municipality-level turnout, party shares, ENP and victory margins.


# Phase 1 continuation — Steps 4–6

The cells below continue from the standardised datasets created above and complete the historical election-data backbone.

**Important:** the historical crosswalk is reproduced from the teammate prototype. It makes the workflow reproducible, but the mapping must be independently verified and documented before the final submission.


In [ ]:
# Step 4 — historical municipality crosswalk
CODE_MAP = {
    "EC103": "EC101", "EC107": "EC101",
    "EC125": "BUF",
    "EC127": "EC129", "EC128": "EC129",
    "EC132": "EC139", "EC133": "EC139", "EC134": "EC139",
    "EC143": "EC145", "EC144": "EC145",
    "EC151": "EC443", "EC152": "EC444",
    "EC05B2": "EC442", "EC05B3": "EC441",
}

TARGET_CODES = {
    "BUF","NMA","EC101","EC102","EC104","EC105","EC106","EC108","EC109",
    "EC121","EC122","EC123","EC124","EC126","EC129","EC131","EC135","EC136",
    "EC137","EC138","EC139","EC141","EC142","EC145","EC153","EC154","EC155",
    "EC156","EC157","EC441","EC442","EC443","EC444"
}

def historical_code(value):
    if pd.isna(value):
        return None
    text = str(value).strip()
    up = text.upper()
    if up.startswith("ECDMA"):
        return None
    if up.startswith("PORT ELIZABETH"):
        return "NMA"
    return text.split(" - ")[0].strip().upper()

crosswalk = pd.DataFrame(
    [{"HistoricalCode": old, "TargetCode": new,
      "Source": "teammate prototype", "VerificationStatus": "REQUIRES VERIFICATION"}
     for old, new in sorted(CODE_MAP.items())]
)
display(crosswalk)
crosswalk.to_csv(PROCESSED_DIR / "municipality_crosswalk_used.csv", index=False)


In [ ]:
# Apply the crosswalk to the standardised row-level data.
harmonised = {}

for year, d in standardised.items():
    x = d.copy()
    x["HistoricalCode"] = x["Municipality"].map(historical_code)
    x["MuniCode"] = x["HistoricalCode"].map(
        lambda c: CODE_MAP.get(c, c) if c is not None else None
    )

    unmapped = (
        x.loc[~x["MuniCode"].isin(TARGET_CODES), ["Municipality", "HistoricalCode"]]
        .drop_duplicates()
        .sort_values(["HistoricalCode", "Municipality"], na_position="last")
    )
    if len(unmapped):
        print("\n", year, "unmapped/dropped labels")
        display(unmapped.head(100))

    x = x[x["MuniCode"].isin(TARGET_CODES)].copy()
    harmonised[year] = x
    print(year, "mapped municipalities:", x["MuniCode"].nunique())


## Step 5 — Municipality-level aggregation and political competition

Registered-voter and spoilt-vote values repeat across party rows, so they are counted once per voting district. Party votes are summed across party rows.

ENP and victory margin are calculated from the **raw party distribution**, before any minor parties are collapsed into an `Other` group.


In [ ]:
def pr_rows(d):
    ballot = d["BallotType"].astype(str).str.strip().str.upper()
    return d[ballot.eq("PR")].copy()

def aggregate_election(d, year):
    pr = pr_rows(d).dropna(subset=["MuniCode", "PartyName", "PartyVotes"]).copy()

    # Count voter-roll / spoilt values once per voting district within the target municipality.
    stations = pr.drop_duplicates(subset=["MuniCode", "VotingDistrict"])
    registered = stations.groupby("MuniCode")["RegisteredVoters"].sum(min_count=1)
    spoilt = stations.groupby("MuniCode")["SpoiltVotes"].sum(min_count=1)

    party = (
        pr.groupby(["MuniCode", "PartyName"], as_index=False)["PartyVotes"]
          .sum()
    )
    party["ValidVotes"] = party.groupby("MuniCode")["PartyVotes"].transform("sum")
    party["PartyShare"] = party["PartyVotes"] / party["ValidVotes"]

    valid = party.groupby("MuniCode")["PartyVotes"].sum()

    municipality = pd.DataFrame({
        "RegisteredVoters": registered,
        "SpoiltVotes": spoilt,
        "ValidVotes": valid,
    })
    municipality["VotesCast"] = municipality["ValidVotes"] + municipality["SpoiltVotes"]
    municipality["Turnout"] = municipality["VotesCast"] / municipality["RegisteredVoters"] * 100

    comp_rows = []
    for code, g in party.groupby("MuniCode"):
        shares = g["PartyShare"].sort_values(ascending=False).to_numpy()
        comp_rows.append({
            "MuniCode": code,
            "NumberOfParties": len(shares),
            "TopShare": shares[0] * 100 if len(shares) else np.nan,
            "VictoryMargin": ((shares[0] - shares[1]) * 100
                              if len(shares) > 1 else shares[0] * 100),
            "ENP": 1 / np.sum(shares ** 2) if len(shares) else np.nan,
        })

    competition = pd.DataFrame(comp_rows).set_index("MuniCode")
    municipality = municipality.join(competition)
    municipality["Year"] = year
    municipality.index.name = "MuniCode"
    municipality = municipality.reset_index()

    party["Year"] = year
    return municipality, party

municipality_tables = {}
party_tables = {}

for year, d in harmonised.items():
    municipality_tables[year], party_tables[year] = aggregate_election(d, year)
    print(year, municipality_tables[year].shape)
    display(municipality_tables[year].head())


In [ ]:
# Combine all five elections.
election_panel = (
    pd.concat(municipality_tables.values(), ignore_index=True)
      .sort_values(["MuniCode", "Year"])
      .reset_index(drop=True)
)

party_long = pd.concat(party_tables.values(), ignore_index=True)

party_wide = (
    party_long.pivot_table(
        index=["MuniCode", "Year"],
        columns="PartyName",
        values="PartyShare",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)

party_columns = [c for c in party_wide.columns if c not in ["MuniCode", "Year"]]
party_wide[party_columns] = party_wide[party_columns] * 100

print("Election panel:", election_panel.shape)
print("Party long:", party_long.shape)
print("Party wide:", party_wide.shape)


## Step 6 — Historical lag features and validation

In [ ]:
# Lag features needed later for temporal modelling.
lag_columns = [
    "RegisteredVoters", "Turnout", "NumberOfParties",
    "TopShare", "VictoryMargin", "ENP"
]

for c in lag_columns:
    election_panel["Previous" + c] = (
        election_panel.groupby("MuniCode")[c].shift(1)
    )

election_panel["TurnoutChange"] = (
    election_panel["Turnout"] - election_panel["PreviousTurnout"]
)
election_panel["ENPChange"] = (
    election_panel["ENP"] - election_panel["PreviousENP"]
)
election_panel["MarginChange"] = (
    election_panel["VictoryMargin"] - election_panel["PreviousVictoryMargin"]
)

display(election_panel.head(15))


In [ ]:
validation_rows = []

for year in YEARS:
    if year not in municipality_tables:
        validation_rows.append({
            "Year": year, "Municipalities": 0, "MissingTargetCodes": 33,
            "BadTurnout": np.nan, "ENPBelow1": np.nan, "BadMargin": np.nan,
            "Status": "SOURCE NOT LOADED"
        })
        continue

    m = municipality_tables[year]
    codes = set(m["MuniCode"])
    missing = sorted(TARGET_CODES - codes)

    bad_turnout = int((~m["Turnout"].between(0, 100)).sum())
    bad_enp = int((m["ENP"] < 1).sum())
    bad_margin = int((~m["VictoryMargin"].between(0, 100)).sum())

    validation_rows.append({
        "Year": year,
        "Municipalities": m["MuniCode"].nunique(),
        "MissingTargetCodes": len(missing),
        "BadTurnout": bad_turnout,
        "ENPBelow1": bad_enp,
        "BadMargin": bad_margin,
        "Status": "PASS" if not (missing or bad_turnout or bad_enp or bad_margin) else "CHECK"
    })
    print(year, "missing codes:", missing)

validation = pd.DataFrame(validation_rows)
display(validation)

assert not election_panel.duplicated(["MuniCode", "Year"]).any()
assert (election_panel["RegisteredVoters"].dropna() >= 0).all()
assert election_panel["Turnout"].dropna().between(0, 100).all()
assert (election_panel["ENP"].dropna() >= 1 - 1e-9).all()
assert election_panel["VictoryMargin"].dropna().between(0, 100).all()

print("Core Phase 1 sanity checks passed.")


In [ ]:
# Diagnostic for possible mapping/boundary issues.
registration_diagnostic = election_panel[
    ["MuniCode", "Year", "RegisteredVoters"]
].copy()

registration_diagnostic["PreviousRegistered"] = (
    registration_diagnostic.groupby("MuniCode")["RegisteredVoters"].shift(1)
)
registration_diagnostic["RegistrationChangePct"] = (
    (registration_diagnostic["RegisteredVoters"] /
     registration_diagnostic["PreviousRegistered"] - 1) * 100
)

print("Largest decreases:")
display(registration_diagnostic.sort_values("RegistrationChangePct").head(15))

print("Largest increases:")
display(registration_diagnostic.sort_values("RegistrationChangePct", ascending=False).head(15))


In [ ]:
# Save the completed Phase 1 products.
FINAL_DIR = PROCESSED_DIR / "elections"
FINAL_DIR.mkdir(parents=True, exist_ok=True)

election_panel.to_csv(FINAL_DIR / "municipality_election_panel_2000_2021.csv", index=False)
party_long.to_csv(FINAL_DIR / "party_shares_long_2000_2021.csv", index=False)
party_wide.to_csv(FINAL_DIR / "party_shares_wide_2000_2021.csv", index=False)
validation.to_csv(FINAL_DIR / "phase1_validation_summary.csv", index=False)
registration_diagnostic.to_csv(FINAL_DIR / "registration_change_diagnostic.csv", index=False)

print("Phase 1 outputs saved to:", FINAL_DIR)
for p in sorted(FINAL_DIR.glob("*.csv")):
    print(" -", p.name)


# Phase 1 completion gate

Before calling the election backbone final for the report:

- verify the historical municipality crosswalk with an authoritative source;
- inspect all dropped/unmapped 2000 and 2006 areas, especially DMA records;
- compare 2016 and 2021 turnout against the official municipality turnout reports already downloaded;
- document municipalities affected by material boundary changes;
- keep the raw-party ENP calculation unchanged.

After those checks, move to **Phase 2: demographic acquisition and temporal alignment**.
